In [1]:
import numpy as np
import matplotlib.pyplot as plt
from SG_solver import rbf_matrix, second_divided_difference, d2_L_W2

import os
import wandb
from tqdm import tqdm

from class_sweep_config import sweep_config

In [2]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/amar-kumar/.netrc
wandb: Currently logged in as: amar743840 (fractal_on_pde) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
a = -1.0
b = 1.0
#######################
# K = 8
# s = 0.8
# c = 0.027 
# ################
# h = 0.01
# tau = 0.01
################
T = 0.25



def exact_u(x, t):
    return 0.5 * (np.sin(np.pi * (x + t)) + np.sin(np.pi * (x - t)))

# time =[0.25]  # , 0.5, 0.75, 1
# for T in tqdm(time, desc="Time stepping", unit="step"):
        
def classical_optimization():
    # Initialize a new wandb run
    # with wandb.init() as run:
    # Get the hyperparameters for this run
    wandb.init()
    config = wandb.config

    run_name = f"c-{config.c}_h-{config.h}_s-{config.s}_K-{config.K}_tau-{config.tau}"
    wandb.run.name = run_name
    # wandb.run.save()

    # Extract hyperparameters from the config
    c = config.c
    s = config.s
    h = config.h
    K = config.K
    tau = config.tau

    n = round((b - a) / h)
    if not np.isclose(a + n*h, b):
        raise ValueError("h must divide b-a exactly.")
    n = int(n)


    if n % K != 0:
        raise ValueError("n must be divisible by K because the paper defines N = n / K.")

    N = n // K
    Nt = int(round(T / tau))
    # Nt = 0

    x = np.linspace(a, b, n + 1)

    # Interpolation center indices k_j, j = 1,...,N
    k_idx = np.concatenate(([1], K * np.arange(1, N - 1), [n - 1])).astype(int)

    xk = x[k_idx]
    xkj = []
    for j in range(1, N+1):
        if j == 1:
            value = float(x[1])
            xkj.append(value)

        elif 1 < j < N:
            value = a + ((j - 1) * (b - a)) / N
            xkj.append(value)
        elif j == N:
            value = float(x[n - 1])
            xkj.append(value)

    U = np.zeros((Nt + 2, len(x)))
    ## Initial condition at t=0
    U[0, :] = exact_u(x, 0)
    U[0, 0]  = 0.0
    U[0, -1] = 0.0

    # Build approximation of u_xx at t=0
    A0 = rbf_matrix(xk, s)
    rhs0 = []
    for kj in k_idx:
        rhs0.append(second_divided_difference(x, U[0, :], kj))
    rhs0 = np.asarray(rhs0)

    alpha0 = np.linalg.solve(A0, rhs0)

    # for 1st time step
    uxx0 = np.zeros(len(x))
    for i in range(len(x)):
        uxx0[i] = d2_L_W2(i, x, U[0, :], xk, alpha0, s, c)
        # uxx0[i] = d2_fractal_L_W2(i, x, U[0, :], xk, alpha0, s, c, f_alpha, n_points=2000, n_iter=50)

    # Since g(x)=0
    U[1, :] = U[0, :] + 0.5 * tau**2 * uxx0
    U[1, 0]  = 0.0
    U[1, -1] = 0.0

    #------------------------------------
    for d in tqdm(range(1, Nt + 1), desc="Time stepping", unit="step"):
        A = rbf_matrix(xk, s)
        rhs_d = np.asarray([second_divided_difference(x, U[d], kj) for kj in k_idx])
        alpha = np.linalg.solve(A, rhs_d)

        uxx = np.zeros(len(x))
        for i in range( len(x)):
            uxx[i] = d2_L_W2(i, x, U[d], xk, alpha, s, c)


        U[d + 1, :] = 2.0 * U[d, :] - U[d - 1, :] + tau**2 * uxx

        # Dirichlet boundary conditions
        U[d + 1, 0] = 0.0
        U[d + 1, -1] = 0.0
    #-----------------------------------

    u_num = U[Nt + 1, :]
    u_ex = exact_u(x, T )  # Exact solution at T+tau because of the central difference initialization)

    err = -(u_ex - u_num)

    abs_err = np.abs(err)

    Linf_error = np.max(abs_err)

    RMS_error = np.sqrt(np.mean(abs_err**2))

    # Log the Linf error to wandb
    wandb.log({"c":c, 
                "s": s,
                "K": K,
                "h": h,
                "tau": tau, 
                "RMS_error": RMS_error,
                "Linf_error": Linf_error})
    


# ---------------------------------------------------
# Run sweep
# ---------------------------------------------------
if __name__ == "__main__":
    sweep_id = wandb.sweep(sweep_config, project="SG_classical_optimization")
    wandb.agent(sweep_id, function=classical_optimization, count = 200)
    print("Sweep complete")

Create sweep with ID: y9oezoej
Sweep URL: https://wandb.ai/fractal_on_pde/SG_classical_optimization/sweeps/y9oezoej


wandb: Agent Starting Run: m6i97f7e with config:
wandb: 	K: 5
wandb: 	c: 0.025
wandb: 	h: 0.02
wandb: 	s: 0.9
wandb: 	tau: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/amar-kumar/.netrc.


Time stepping: 100%|██████████| 25/25 [00:02<00:00, 10.08step/s]


K,▁
Linf_error,▁
RMS_error,▁
c,▁
h,▁
s,▁
tau,▁
K,5
Linf_error,0.02258
RMS_error,0.01589
c,0.025


wandb: Agent Starting Run: d549lp5s with config:
wandb: 	K: 10
wandb: 	c: 0.035
wandb: 	h: 0.1
wandb: 	s: 0.8
wandb: 	tau: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/amar-kumar/.netrc.


...

wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: dgc0qe27 with config:
wandb: 	K: 4
wandb: 	c: 0.035
wandb: 	h: 0.02
wandb: 	s: 0.6
wandb: 	tau: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/amar-kumar/.netrc.


Time stepping: 100%|██████████| 25/25 [00:03<00:00,  7.21step/s]


K,▁
Linf_error,▁
RMS_error,▁
c,▁
h,▁
s,▁
tau,▁
K,4
Linf_error,0.02258
RMS_error,0.01589
c,0.035


wandb: Sweep Agent: Waiting for job.
wandb: Ctrl + C detected. Stopping sweep.


Sweep complete
